# DPPUv2 paper03ec: Paper Figures
## Homogeneous Three-Topology Comparison in EC-Weyl Theory

This notebook generates three figures for paper03ec using the SymPy EC engine.

### Figure map
| Figure | Section | Content | Cell |
|--------|---------|---------|------|
| Fig. 1 | §5.2 (Theorem 6) | Nil³ effective potential $V_{\rm eff}(r)$ and EC slice minimum | Engine, Fig1 |
| Fig. 2 | §5.3 (Theorem 7) | Spin-2 quintet splitting $5\to 0+2+1+1$ | Engine, Fig2 |
| Fig. 3 | §6.3 | $\gamma=1/2$ scaling: $r_0$ vs $\sqrt{\|\alpha\|}$ | Engine, Fig3 |

**Author:** Muacca  
**Date:** 2026-04-02

---
> **Computation time:**
> - `[Engine]` cell (first run): ~20 s total (3 SymPy engines).
> - Subsequent runs: instant from `output/paper03ec/figures_cache.pkl`.
> - Figure cells: instant.

**Engine map:**

| Engine config | Purpose | Time |
|---------------|---------|------|
| Nil³ AX + squash + shear | $V_{\rm eff}(r,\varepsilon,s;\alpha)$ → Figs 1, 2, 3 | ~3 s |
| Nil³ off-diagonal shear | $m^2(q_3), m^2(q_4=q_5)$ via $z_3,z_4,z_5$ DOFs | ~14 s |
| Nil³ TWIST (anisotropic) | $m^2(\omega_k)$ spin-1 masses | ~0.1 s |

In [ ]:
import sys
import os
import pickle
import time

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize_scalar

# ── Matplotlib style ──────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.dpi': 110,
    'savefig.dpi': 200,
})

# ── Path setup ─────────────────────────────────────────────────
# Notebook lives at scripts/visualize/ — project root is ../..
notebook_dir = os.path.abspath('')
project_root = os.path.normpath(os.path.join(notebook_dir, '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f'Project root: {project_root}')

output_dir = os.path.join(project_root, '../data')
cache_file  = os.path.join(output_dir, 'figures_cache.pkl')
os.makedirs(output_dir, exist_ok=True)

# ── Cache helpers ──────────────────────────────────────────────
def load_cache():
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    return {}

def save_cache(c):
    with open(cache_file, 'wb') as f:
        pickle.dump(c, f)

cache = load_cache()
print(f'Cached keys : {list(cache.keys())}')
print(f'Output dir  : {output_dir}')

## [Engine] SymPy engine computation

Runs three SymPy engines for Nil³×S¹ and computes all data for Figs 1, 2, 3.
First run ~20 s; subsequent runs load from cache instantly.

**Engine 1 — Nil³ AX + squash + shear** (`~3 s`)
Computes $V_{\rm eff}(r,\varepsilon,s;\alpha)$ with $\kappa=L=1$, $\eta=0$.
Used for: the $V_{\rm eff}(r;\alpha)$ curves (Fig. 1),
the squash/shear/cross Hessian (Fig. 2), and the $r_0(\alpha)$ scan (Fig. 3).

**Engine 2 — Nil³ off-diagonal shear** (`~14 s`)
Adds off-diagonal vielbein deformation DOFs $z_3, z_4, z_5$
(hyperbolic rotations; identity at $z_i=1$).
Symbolic second derivatives $\partial^2 V/\partial z_i^2|_{z_i=1}$ give
$m^2(q_3)=0$ (geometric zero mode) and $m^2(q_4)=m^2(q_5)$ (doublet).

**Engine 3 — Nil³ TWIST (anisotropic)** (`~0.1 s`)
Computes spin-1 twist DOFs $\omega_1,\omega_2,\omega_3$.
$\omega_3$ (engine index) = $\omega_2$ (paper notation, non-commutative axis)
acquires mass; $\omega_1=\omega_2=0$ (commutative axes).

In [ ]:
# ── [Engine] Nil3 EC mass spectrum and V_eff scan ─────────────────────────
# First run ~20s. Subsequent runs: instant (cache).

NEED_VEFF   = 'nil3_veff'   not in cache
NEED_MASSES = 'nil3_masses' not in cache

if NEED_VEFF or NEED_MASSES:
    print('[ENGINE] Running SymPy engines...')
    t0_total = time.time()

    from sympy import cancel, diff, lambdify, S
    from dppu.topology.unified import UnifiedEngine, DOFConfig, TopologyType, FiberMode
    from dppu.torsion.mode import Mode
    from dppu.torsion.nieh_yan import NyVariant

    # ── Engine 1: Nil3 AX + squash + shear ─────────────────────
    print('  [1/3] Nil3 AX squash+shear...', end='', flush=True)
    t1 = time.time()
    cfg1 = DOFConfig(
        topology=TopologyType.NIL3,
        torsion_mode=Mode.AX,
        enable_squash=True,
        enable_shear=True,
        ny_variant=NyVariant.FULL,
    )
    eng1 = UnifiedEngine(cfg1)
    eng1.run()
    p1 = eng1.data['params']
    V1 = eng1.data['potential']
    r_s, eps_s, s_s = p1['r'], p1['epsilon'], p1['s']
    eta_s, alpha_s, L_s, kappa_s = p1['eta'], p1['alpha'], p1['L'], p1['kappa']
    print(f' done ({time.time()-t1:.1f}s)')

    # Isotropic slice V(r; alpha): kappa=L=1, eta=eps=s=0
    V_iso = cancel(V1.subs([(eta_s,0),(eps_s,0),(s_s,0),(L_s,1),(kappa_s,1)]))
    print(f'  V_iso = {V_iso}')
    V_fn = lambdify((r_s, alpha_s), V_iso, modules='numpy')

    # Full V(r, eps, s; alpha): kappa=L=1, eta=0
    V_sq_sym = cancel(V1.subs([(eta_s,0),(L_s,1),(kappa_s,1)]))
    V_sq_fn  = lambdify((r_s, eps_s, s_s, alpha_s), V_sq_sym, modules='numpy')

    # ── Figs 1 & 3: r0 scan over alpha ─────────────────────────
    r_arr   = np.linspace(0.5, 7.5, 400)
    alphas  = [0.0, -0.5, -1.0, -3.0]
    V_curves = {}
    r0_vals  = {}
    V0_vals  = {}
    for a in alphas:
        V_curves[str(a)] = V_fn(r_arr, a).tolist()
        if a < 0:
            res = minimize_scalar(
                lambda r: float(V_fn(r, a)),
                bounds=(0.1, 9.0), method='bounded')
            r0_vals[str(a)] = float(res.x)
            V0_vals[str(a)] = float(V_fn(res.x, a))

    # Fig 3: fine alpha scan for gamma=1/2 fit
    alpha_abs_scan = np.array([0.05, 0.10, 0.25, 0.50, 1.00, 2.00, 3.00])
    r0_scan = np.array([
        minimize_scalar(lambda r, a=a: float(V_fn(r, -a)),
                        bounds=(0.1, 10.0), method='bounded').x
        for a in alpha_abs_scan
    ])
    V0_scan = np.array([float(V_fn(r0_scan[i], -alpha_abs_scan[i]))
                        for i in range(len(alpha_abs_scan))])

    if NEED_VEFF:
        cache['nil3_veff'] = dict(
            V_iso_repr=str(V_iso),
            r_arr=r_arr.tolist(),
            alphas=alphas,
            V_curves=V_curves,
            r0_vals=r0_vals,
            V0_vals=V0_vals,
            alpha_abs_scan=alpha_abs_scan.tolist(),
            r0_scan=r0_scan.tolist(),
            V0_scan=V0_scan.tolist(),
        )
        save_cache(cache)
        print(f'  nil3_veff cached ({time.time()-t0_total:.1f}s elapsed)')

    if NEED_MASSES:
        # ── Fig 2: Hessian at EC slice minimum (alpha=-1) ────────
        alpha_val = -1.0
        r0 = r0_vals['-1.0']
        print(f'  r0(alpha=-1) = {r0:.4f}')

        h = 1e-4
        # (a) r-mass
        m2_r = (V_fn(r0+h, alpha_val) + V_fn(r0-h, alpha_val)
                - 2*V_fn(r0, alpha_val)) / h**2
        # (b) squash/shear Hessian
        V0c   = V_sq_fn(r0, 0, 0, alpha_val)
        m2_eps = (V_sq_fn(r0, +h, 0, alpha_val) + V_sq_fn(r0, -h, 0, alpha_val) - 2*V0c) / h**2
        m2_s   = (V_sq_fn(r0, 0, +h, alpha_val) + V_sq_fn(r0, 0, -h, alpha_val) - 2*V0c) / h**2
        m2_cross = (  V_sq_fn(r0, +h, +h, alpha_val)
                    - V_sq_fn(r0, +h, -h, alpha_val)
                    - V_sq_fn(r0, -h, +h, alpha_val)
                    + V_sq_fn(r0, -h, -h, alpha_val)) / (4*h**2)
        M2 = np.array([[m2_eps, m2_cross],[m2_cross, m2_s]])
        lams = np.linalg.eigvalsh(M2)
        print(f'  2x2 block eigenvalues: lam- = {lams[0]:.4e}, lam+ = {lams[1]:.4e}')

        # (c) LC reference squash mass (alpha=0, same r0)
        m2_lc = (V_sq_fn(r0, +h, 0, 0.0) + V_sq_fn(r0, -h, 0, 0.0)
                 - 2*V_sq_fn(r0, 0, 0, 0.0)) / h**2
        print(f'  m2_lc(alpha=0, r=r0) = {m2_lc:.4e}')

        # ── Engine 2: off-diagonal DOFs z3/z4/z5 ────────────────
        print('  [2/3] Nil3 off-diagonal shear...', end='', flush=True)
        t2 = time.time()
        cfg2 = DOFConfig(
            topology=TopologyType.NIL3,
            torsion_mode=Mode.AX,
            enable_offdiag_shear=True,
            ny_variant=NyVariant.FULL,
        )
        eng2 = UnifiedEngine(cfg2)
        eng2.run()
        p2 = eng2.data['params']
        V2 = eng2.data['potential']
        z3, z4, z5   = p2['z3'], p2['z4'], p2['z5']
        q3, q4, q5   = p2['q3'], p2['q4'], p2['q5']
        r2, L2, kap2 = p2['r'], p2['L'], p2['kappa']
        eta2, alp2   = p2['eta'], p2['alpha']
        eps2, s2     = p2['epsilon'], p2['s']
        print(f' done ({time.time()-t2:.1f}s)')

        # Background: all DOFs at identity/zero, kappa=L=1, eta=0
        base2 = [
            (L2,1),(kap2,1),(eta2,0),(eps2,0),(s2,0),
            (q3,0),(q4,0),(q5,0),
            (p2['delta0'],0),(p2['delta1'],0),(p2['delta2'],0),
            (p2['omega1'],0),(p2['omega2'],0),(p2['omega3'],0),
            (p2.get('V',S.Zero),0),(p2.get('q',S.Zero),0),
            (alp2, alpha_val),(r2, r0),
        ]

        # d2V/dz_i^2 at z_i=1 (hyperbolic identity), others=1
        # Mass = d2V/dz_i^2 / 2  (kinematic normalization from nil3_spin2_quintet_splitting.py)
        m2_offdiag = {}
        for zi, label in [(z3,'q3'),(z4,'q4'),(z5,'q5')]:
            subs_i = base2 + [(zj,1) for zj in (z3,z4,z5) if zj is not zi]
            d2_sym = diff(diff(V2, zi), zi)
            d2_val = cancel(d2_sym.subs(subs_i + [(zi,1)]))
            m2_offdiag[label] = float(d2_val) / 2
            print(f'  d2V/d{zi}^2 / 2 = {m2_offdiag[label]:.4e}  [{label}]')

        # ── Engine 3: spin-1 omega masses ────────────────────────
        print('  [3/3] Nil3 TWIST...', end='', flush=True)
        t3 = time.time()
        cfg3 = DOFConfig(
            topology=TopologyType.NIL3,
            torsion_mode=Mode.AX,
            fiber_mode=FiberMode.TWIST,
            isotropic_twist=False,
            ny_variant=NyVariant.FULL,
        )
        eng3 = UnifiedEngine(cfg3)
        eng3.run()
        p3 = eng3.data['params']
        V3 = eng3.data['potential']
        r3, L3, kap3 = p3['r'], p3['L'], p3['kappa']
        eta3, alp3   = p3['eta'], p3['alpha']
        om1, om2, om3 = p3['omega1'], p3['omega2'], p3['omega3']
        print(f' done ({time.time()-t3:.1f}s)')

        # d2V/dom_k^2 at om_k=0, kappa=L=1, eta=0
        base3 = [(L3,1),(kap3,1),(eta3,0),(om1,0),(om2,0),(om3,0),
                 (alp3, alpha_val),(r3, r0)]
        m2_spin1 = {}
        for omi, name in [(om1,'omega1'),(om2,'omega2'),(om3,'omega3')]:
            subs_i = [(k,v) for k,v in base3 if k is not omi]
            d2_sym = diff(diff(V3, omi), omi)
            d2_val = cancel(d2_sym.subs(subs_i + [(omi,0)]))
            m2_spin1[name] = float(d2_val)
        print(f'  spin-1 masses: {m2_spin1}')
        print('  (omega3 in engine = omega_2 in paper: non-commutative axis)')

        cache['nil3_masses'] = dict(
            alpha_val=alpha_val,
            r0=float(r0),
            m2_r=float(m2_r),
            m2_eps=float(m2_eps),
            m2_s=float(m2_s),
            m2_cross=float(m2_cross),
            lam_minus=float(lams[0]),
            lam_plus=float(lams[1]),
            m2_lc=float(m2_lc),
            m2_q3=m2_offdiag['q3'],
            m2_q4=m2_offdiag['q4'],
            m2_q5=m2_offdiag['q5'],
            m2_omega1=m2_spin1['omega1'],
            m2_omega2=m2_spin1['omega2'],
            m2_omega3=m2_spin1['omega3'],
        )
        save_cache(cache)
        print(f'  nil3_masses cached.')

    print(f'[ENGINE] Done. Total: {time.time()-t0_total:.1f}s.')

else:
    print('Loading all data from cache...')

# ── Unpack cache ─────────────────────────────────────────────
nd = cache['nil3_veff']
nm = cache['nil3_masses']

r_arr    = np.array(nd['r_arr'])
V_curves = {float(k): np.array(v) for k, v in nd['V_curves'].items()}
r0_vals  = {float(k): v for k, v in nd['r0_vals'].items()}
V0_vals  = {float(k): v for k, v in nd['V0_vals'].items()}
alpha_abs_scan = np.array(nd['alpha_abs_scan'])
r0_scan        = np.array(nd['r0_scan'])
V0_scan        = np.array(nd['V0_scan'])

print(f"V_iso = {nd['V_iso_repr']}")
print(f"r0(alpha=-1) = {nm['r0']:.4f}")
pi = np.pi
print()
print('Mass spectrum at EC slice minimum (alpha=-1, kappa=L=1):')
for key in ['m2_r','m2_eps','m2_s','m2_cross','lam_minus','lam_plus',
            'm2_lc','m2_q3','m2_q4','m2_q5',
            'm2_omega1','m2_omega2','m2_omega3']:
    print(f'  {key:20s} = {nm[key]:.4e}')

## Fig. 1 — §5.2: Nil³ effective potential and EC slice minimum

**Source:** EC engine (Engine 1)  
**Section:** §5.2 (Theorem 6)

The engine yields the Nil³×S¹ effective potential on the $\eta=V=0$ isotropic slice
($\kappa=L=1$):

$$
V_{\rm eff}^{\,Nil^3}(r;\,\alpha) = \frac{12\pi^4 R^2 - 64\pi^4\alpha}{3R}
\quad \Bigl(= 4\pi^4 r - \tfrac{64\pi^4}{3}\tfrac{\alpha}{r}\Bigr).
$$

The engine expression $(12\pi^4 R^2 - 64\pi^4\alpha)/(3R)$ is identical to the
2-term analytic form of Theorem 6 (printed by the engine cell above).

For $\alpha < 0$ an **EC slice minimum** appears at $r_0 = \partial V/\partial r = 0$,
found numerically via `scipy.optimize.minimize_scalar`.
Each colored dot marks the engine-computed $r_0$ and $V_0=V_{\rm eff}(r_0)$.

In [ ]:
# ── Fig. 1 — Nil³ V_eff(r; alpha) ──────────────────────────────────────────
plt.close('all')

alphas_plot = [0.0, -0.5, -1.0, -3.0]
colors = ['#555555', '#2196F3', '#F44336', '#9C27B0']
labels = [
    r'$\alpha = 0$ (LC, monotone)',
    r'$\alpha = -0.5$',
    r'$\alpha = -1$',
    r'$\alpha = -3$',
]

fig, ax = plt.subplots(figsize=(7.5, 5))

for alpha, color, label in zip(alphas_plot, colors, labels):
    y = V_curves[alpha] / pi**4
    ax.plot(r_arr, y, color=color, lw=2, label=label)
    if alpha < 0:
        r0v = r0_vals[alpha]
        V0v = V0_vals[alpha] / pi**4
        ax.plot(r0v, V0v, 'o', color=color, ms=8, zorder=5,
                markeredgecolor='white', markeredgewidth=0.8)
        # Vertical guide
        ax.axvline(r0v, color=color, lw=0.7, ls='--', alpha=0.35)

# Annotate r0 formula (alpha=-1)
r0_m1 = r0_vals[-1.0]
V0_m1 = V0_vals[-1.0] / pi**4
ax.annotate(
    r'$r_0(\alpha)\!=\!\frac{4}{\sqrt{3}}\sqrt{|\alpha|}$',
    xy=(r0_m1, V0_m1), xytext=(r0_m1 + 1.0, V0_m1 + 3.5),
    arrowprops=dict(arrowstyle='->', color='#F44336', lw=1.4),
    fontsize=11, color='#F44336', ha='left',
)

ax.axhline(0, color='gray', lw=0.6, ls=':')
ax.set_xlabel(r'Scale parameter $r$  ($\kappa = L = 1$)', fontsize=12)
ax.set_ylabel(r'$V_{\rm eff}^{\,Nil^3}(r)\;/\;\pi^4$', fontsize=12)
ax.set_title(
    r'Nil$^3\times S^1$ effective potential — EC engine (Theorem 6)',
    fontsize=12)
ax.set_xlim(0.5, 7.5)
ax.set_ylim(-1, 42)
ax.legend(loc='upper right', framealpha=0.9)
ax.grid(True, alpha=0.25)

plt.tight_layout()
out_path = os.path.join(output_dir, 'fig01_nil3_veff.png')
plt.savefig(out_path, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

# ── Print engine-computed EC slice-minimum table ───────────────
print('\nEngine-computed EC slice-minimum table (kappa=L=1):')
print(f'  {"alpha":>7}  {"r0":>10}  {"V0/pi^4":>12}')
for a in sorted(r0_vals.keys()):
    print(f'  {a:>7.2f}  {r0_vals[a]:>10.6f}  {V0_vals[a]/pi**4:>12.6f}')

## Fig. 2 — §5.3: Nil³ spin-2 quintet splitting $5\to 0+2+1+1$

**Source:** EC engines (Engines 1–3)  
**Section:** §5.3 (Theorem 7)

All 8 mass values come from the engine cells:

| DOF | Engine | Method | Expected |
|-----|--------|--------|----------|
| $r$ (radial) | Engine 1 | numerical $\partial^2 V/\partial r^2$ | $\propto 1/a$ |
| $\varepsilon$ (squash) | Engine 1 | numerical $\partial^2 V/\partial\varepsilon^2$ | $3.919\times10^4$ |
| $s$ (shear) | Engine 1 | from trace $\lambda_++\lambda_--m^2_\varepsilon$ | $8.278\times10^4$ |
| $\varepsilon$–$s$ cross | Engine 1 | numerical mixed deriv. | $4.799\times10^4$ |
| $\lambda_\pm$ | Engine 1 | `np.linalg.eigvalsh` of $2\times2$ block | $8.28\times10^3$, $1.14\times10^5$ |
| $q_3$ (zero mode) | Engine 2 | $\partial^2 V/\partial z_3^2\|_{z_3=1}/2$ | $0$ |
| $q_4=q_5$ (doublet) | Engine 2 | $\partial^2 V/\partial z_4^2\|_{z_4=1}/2$ | $1.080\times10^4$ |
| $\omega_3$ (non-comm.) | Engine 3 | numerical $\partial^2 V/\partial\omega_3^2$ | $1.012\times10^3$ |

**Note on index mapping:**  
- $z_3$ (engine) $\leftrightarrow$ $q_3$ (paper): $(e^0,e^1)$-plane hyperbolic rotation.
  The zero mode $m^2=0$ follows from $C^2_{01}$ being invariant under this rotation.
- $z_4, z_5$ (engine) $\leftrightarrow$ $q_4,q_5$ (paper): doublet.
- $\omega_3$ (engine, 1-indexed) $\leftrightarrow$ $\omega_2$ (paper): non-commutative axis
  determined by $C^2_{01}\neq 0$.

In [ ]:
# ── Fig. 2 — Nil³ spin-2 quintet splitting (spectroscopic) ──────────────────
plt.close('all')

# ── Unpack mass values from engine cache ─────────────────────────────────────
alpha_val = nm['alpha_val']   # -1.0
r0        = nm['r0']
m2_q3   = nm['m2_q3']         # 0  (geometric zero mode)
m2_q4   = nm['m2_q4']         # doublet
m2_q5   = nm['m2_q5']         # should equal m2_q4 (Nil3 symmetry)
m2_eps  = nm['m2_eps']        # squash
m2_s    = nm['m2_s']          # shear
m2_cross = nm['m2_cross']     # off-diagonal cross term
lam_plus  = nm['lam_plus']    # larger eigenvalue of 2x2 block
lam_minus = nm['lam_minus']   # smaller eigenvalue
m2_lc   = nm['m2_lc']         # LC reference (alpha=0, same r0)
m2_om3  = nm['m2_omega3']     # omega3 (paper: omega_2, non-comm. axis)

print(f'alpha = {alpha_val},  r0 = {r0:.4f}')
print(f'm2(q3)    = {m2_q3:.3e}  [zero mode, engine z3]')
print(f'm2(q4)    = {m2_q4:.3e}  [doublet, engine z4]')
print(f'm2(q5)    = {m2_q5:.3e}  [doublet, engine z5 — should = q4]')
print(f'm2(eps)   = {m2_eps:.4e}  [squash diagonal]')
print(f'm2(s)     = {m2_s:.4e}   [shear diagonal]')
print(f'm2_cross  = {m2_cross:.4e}  [eps-s off-diagonal]')
print(f'lambda+   = {lam_plus:.4e}  [2x2 eigval]')
print(f'lambda-   = {lam_minus:.4e}   [2x2 eigval]')
print(f'm2_lc     = {m2_lc:.4e}  [LC ref, alpha=0]')
print(f'm2(om3)   = {m2_om3:.4e}  [omega_2 non-comm, engine omega3]')
print(f'  q4=q5 check: {abs(m2_q4-m2_q5)/max(abs(m2_q4),1e-8)*100:.3f}% diff')
print(f'  det check: {m2_eps*m2_s - m2_cross**2:.4e} vs {lam_plus*lam_minus:.4e}')

# Use mean of q4/q5 for doublet mass
m2_doublet = 0.5 * (m2_q4 + m2_q5)

# ── Spectroscopic diagram ─────────────────────────────────────────────────────
sc4 = 1e-4   # scale: display in units of 1e4

fig, ax = plt.subplots(figsize=(8.5, 6.5))
ax.set_xlim(-0.05, 1.10)
ax.set_ylim(-0.8, 13.8)

x_lc, x_ec = 0.22, 0.72
hw = 0.09

# ── LC reference: approx 5-plet at same r0 with alpha=0 ──────────────────────
y_lc = m2_lc * sc4
for dy in np.linspace(-0.20, 0.20, 5):
    ax.hlines(y_lc + dy, x_lc - hw, x_lc + hw,
              colors='#888888', lw=1.8, zorder=3)
ax.text(x_lc, y_lc + 1.2,
        r'$5\times$ (isotropic, $\alpha=0$)',
        ha='center', va='bottom', fontsize=10, color='#555555', style='italic')

# ── EC levels ────────────────────────────────────────────────────────────────
ec_modes = [
    (r'$q_3 = 0$',           m2_q3,      1, '#2196F3'),
    (r'$q_4 = q_5$',         m2_doublet, 2, '#FF9800'),
    (r'$\varepsilon$ (squash)', m2_eps,   1, '#4CAF50'),
    (r'$s$ (shear)',          m2_s,       1, '#9C27B0'),
]

for label, mass, deg, color in ec_modes:
    y_val = mass * sc4
    lw_bar = 3.5 if deg == 2 else 2.5
    ax.hlines(y_val, x_ec - hw, x_ec + hw, colors=color, lw=lw_bar, zorder=4)
    if deg == 2:
        ax.hlines(y_val, x_ec - hw, x_ec + hw,
                  colors='white', lw=1.2, ls=(0,(4,3)), zorder=5)
    ax.text(x_ec + hw + 0.015, y_val, label, va='center',
            fontsize=11, color=color)
    if mass > 0:
        ax.text(x_ec + hw + 0.015, y_val - 0.75,
                f'$m^2={mass:.3g}$',
                va='center', fontsize=8.5, color='#888888')
    else:
        ax.text(x_ec + hw + 0.015, y_val - 0.75,
                r'zero mode (geometric)',
                va='center', fontsize=8.5, color='#2196F3', style='italic')
    ax.annotate('',
                xy=(x_ec - hw, y_val), xytext=(x_lc + hw, y_lc),
                arrowprops=dict(arrowstyle='-', color='#BBBBBB', lw=0.9, ls='dashed'))

# Eigenvalue annotations
for lam, label, col in [
        (lam_minus, r'$\lambda_-$', '#4CAF50'),
        (lam_plus,  r'$\lambda_+$', '#9C27B0'),
]:
    ax.axhline(lam*sc4, xmin=x_ec-0.15, xmax=x_ec+0.15,
               color=col, lw=1.0, ls=':', alpha=0.55, zorder=2)
    ax.text(x_ec - hw - 0.02, lam*sc4,
            f'{label}$\\!={lam*sc4:.2f}\\!\\times\\!10^4$',
            va='center', ha='right', fontsize=8.5, color=col, alpha=0.75, style='italic')

# Column headers
ax.axvline(0.5, color='#DDDDDD', lw=1.0)
ax.text(x_lc, 13.3, r'LC ($\alpha=0$, ref)',
        ha='center', fontsize=11, fontweight='bold', color='#555555')
ax.text(x_ec, 13.3, r'EC slice minimum ($\alpha=-1$)',
        ha='center', fontsize=11, fontweight='bold', color='#333333')

ax.set_ylabel(r'$m^2\;[\times 10^4]$   ($\kappa=L=1$, $r_0=4/\!\sqrt{3}$)',
             fontsize=12)
ax.set_title(
    r'Nil$^3\times S^1$ spin-2 quintet splitting (Theorem 7, EC engine)'  '\n'
    r'$5\;\to\;(1)_0 + (2)_{q_4=q_5} + (1)_\varepsilon + (1)_s$',
    fontsize=12)
ax.set_xticks([x_lc, x_ec])
ax.set_xticklabels(['LC\n(isotropic, ref)', 'EC slice minimum\n(Nil$^3$ geometry)'],
                   fontsize=10)
ax.tick_params(axis='x', length=0)
ax.spines[['top','right','bottom']].set_visible(False)
ax.grid(axis='y', alpha=0.18)

plt.tight_layout()
out_path = os.path.join(output_dir, 'fig02_nil3_quintet_splitting.png')
plt.savefig(out_path, bbox_inches='tight')
print(f'\nSaved: {out_path}')
plt.show()

## Fig. 3 — §6.3: $\gamma=1/2$ scaling $r_0 \propto \sqrt{|\alpha|}$

**Source:** EC engine (Engine 1) + least-squares fit  
**Section:** §6.3 ($\gamma=1/2$ scaling theorem)

The engine scans $|\alpha| \in \{0.05, 0.10, 0.25, 0.50, 1.00, 2.00, 3.00\}$
and finds $r_0(\alpha)$ for each via `minimize_scalar`.

Left panel: $r_0$ vs $\sqrt{|\alpha|}$ — the engine points lie exactly on the
analytical line $r_0 = (4/\sqrt{3})\sqrt{|\alpha|}$,
verifying the algebraic exponent $\gamma = n/(n-m) = 1/(1-(-1)) = 1/2$.

Right panel: residuals $r_0^{\rm engine} / r_0^{\rm analytic}$ — confirms machine-precision
agreement (all deviations $< 10^{-4}$).

**Note:** The $\gamma=1/2$ scaling applies to Nil³ only.
$T^3$ (flat: $C^2_{\rm LC}=0$) and $S^3$ (conformally flat: $C^2_{\rm LC}(\eta=V=0)=0$)
yield no isolated EC slice minimum on the $\eta=V=0$ slice (§6.4).

In [ ]:
# ── Fig. 3 — gamma=1/2 scaling: r0 vs sqrt(|alpha|) ─────────────────────────
plt.close('all')

# Analytical line (from Theorem 6)
slope = 4.0 / np.sqrt(3)    # = 4*kappa/sqrt(3), kappa=1
sqrt_a_cont  = np.linspace(0.0, 1.85, 300)
r0_analytic_cont = slope * sqrt_a_cont

# Engine data
sqrt_a_engine  = np.sqrt(alpha_abs_scan)
r0_engine      = r0_scan
r0_analytic_pts = slope * sqrt_a_engine

# Least-squares fit: r0 = k * sqrt(|alpha|)
k_fit = float(np.dot(sqrt_a_engine, r0_engine) / np.dot(sqrt_a_engine, sqrt_a_engine))
print(f'Analytic slope : {slope:.6f}')
print(f'Fitted slope   : {k_fit:.6f}  (from engine data)')
print(f'Relative error : {abs(k_fit/slope - 1)*100:.4f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# ── Left: r0 vs sqrt(|alpha|) ─────────────────────────────────────────────
ax1.plot(sqrt_a_cont, r0_analytic_cont, 'k-', lw=2, zorder=1,
         label=r'$r_0=(4/\!\sqrt{3})\sqrt{|\alpha|}$ (Theorem 6)')
ax1.plot(sqrt_a_cont, k_fit * sqrt_a_cont, '#E91E63', lw=1.5, ls='--',
         zorder=2, alpha=0.8,
         label=fr'Least-squares fit: slope = {k_fit:.4f}')
ax1.scatter(sqrt_a_engine, r0_engine, s=90, c='#F44336', zorder=5,
            edgecolors='white', linewidths=0.8, label='Engine $r_0(\\alpha)$')

ax1.set_xlabel(r'$\sqrt{|\alpha|}$', fontsize=13)
ax1.set_ylabel(r'$r_0$  ($\kappa=L=1$)', fontsize=13)
ax1.set_title(r'EC slice minimum: $r_0$ vs $\sqrt{|\alpha|}$  (engine scan)', fontsize=12)
ax1.legend(fontsize=10, loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1.85)
ax1.set_ylim(0, 4.6)

# Slope annotation
ax1.annotate(
    f'slope $= 4/\\!\\sqrt{{3}} \\approx {slope:.3f}$',
    xy=(1.3, slope*1.3), xytext=(0.7, slope*1.3 + 0.55),
    arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
    fontsize=10, ha='left',)

# ── Right: residuals ──────────────────────────────────────────────────────────
ratio = r0_engine / r0_analytic_pts
ax2.axhline(1.0, color='gray', lw=1.5, ls='--', alpha=0.8,
            label=r'Exact $\gamma=1/2$')
ax2.scatter(sqrt_a_engine, ratio, s=100, c='#F44336', zorder=5,
            edgecolors='white', linewidths=0.8, label='Engine / Analytic')

ax2.set_xlabel(r'$\sqrt{|\alpha|}$', fontsize=13)
ax2.set_ylabel(r'$r_0^{\rm engine}\;/\;r_0^{\rm analytic}$', fontsize=13)
ax2.set_title(r'Residuals (engine vs Theorem 6)', fontsize=12)
ax2.set_xlim(0, 1.85)
y_margin = max(abs(ratio - 1.0)) * 5 + 1e-5
ax2.set_ylim(1 - y_margin, 1 + y_margin)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

max_err = float(np.max(np.abs(ratio - 1.0)))
ax2.text(0.5, 0.07, f'Max deviation: {max_err:.2e}',
         transform=ax2.transAxes, fontsize=10, color='#666666', ha='center')

plt.suptitle(
    r'$\gamma=1/2$ scaling (Nil$^3$ only: $T^3$, $S^3$ conformally flat $\Rightarrow$ no EC vacuum)',
    fontsize=11, y=1.02)
plt.tight_layout()
out_path = os.path.join(output_dir, 'fig03_gamma_scaling.png')
plt.savefig(out_path, bbox_inches='tight')
print(f'\nSaved: {out_path}')
plt.show()

# ── Algebraic confirmation ─────────────────────────────────────────────────────
print()
print('Scaling exponent (algebraic from V_iso structure):')
print(f'  V_iso = {nd["V_iso_repr"]}')
print('  -> n=1 (r term), m=-1 (1/r term)')
print('  -> gamma = n/(n-m) = 1/2  [exact, Theorem 6 / §6.3]')

## Summary

All figures generated from engine computation:

| Figure | File | Section | Engine(s) | Description |
|--------|------|---------|-----------|-------------|
| Fig. 1 | `fig01_nil3_veff.pdf` | §5.2 | Engine 1 | Nil³ $V_{\rm eff}(r)$ with EC false vacua |
| Fig. 2 | `fig02_nil3_quintet_splitting.pdf` | §5.3 | Engines 1–3 | Quintet splitting $5\to0+2+1+1$ |
| Fig. 3 | `fig03_gamma_scaling.pdf` | §6.3 | Engine 1 | $\gamma=1/2$ scaling $r_0$ vs $\sqrt{\|\alpha\|}$ |

Output PNGs: `data/`  
Cache: `data/figures_cache.pkl`

---
**Engine reproducibility:**  
All results are derived from `dppu.topology.unified.UnifiedEngine` with
`TopologyType.NIL3`, `Mode.AX`, and SymPy symbolic computation.
No numerical values are hardcoded — all masses, $r_0$, and $V_0$ are
computed from the engine output.

**Related scripts (full derivation):**
- `scripts/paper03ec/nil3_ec_slice_minimum_eft.py`
- `scripts/paper03ec/nil3_spin2_quintet_splitting.py`
- `scripts/paper03ec/squash_shear_cross_term.py`